In [2]:
import sqlite3
import pandas as pd
from pathlib import Path

DB_PATH = Path(r"C:\Users\maidk\Desktop\Reconciliation Task\EnterpriseReconciliation\enterprise_database.db")

print("Database exists:", DB_PATH.exists())

with sqlite3.connect(DB_PATH) as conn:
    tables = pd.read_sql_query(
        """
        SELECT name
        FROM sqlite_master
        WHERE type = 'table'
        """,
        conn
    )

display(tables)

Database exists: True


,name
0,users


In [3]:
from pathlib import Path

print("Notebook working directory:")
print(Path.cwd())

print("\nDatabase files found:")
for file in Path.cwd().glob("*.db"):
    print(file.name, file.stat().st_size, "bytes")

Notebook working directory:
C:\Users\maidk\Desktop\Reconciliation Task\EnterpriseReconciliation

Database files found:
enterprise_database (1).db 0 bytes
enterprise_database.db 2670592 bytes


In [4]:
with sqlite3.connect(DB_PATH) as conn:
    columns = pd.read_sql_query(
        "PRAGMA table_info(users)",
        conn
    )

display(columns)

,cid,name,type,notnull,dflt_value,pk
0,0,user_id,TEXT,0,None,0
1,1,name,TEXT,0,None,0
2,2,status,TEXT,0,None,0
3,3,updated_at,TIMESTAMP,0,None,0


In [5]:
with sqlite3.connect(DB_PATH) as conn:
    customers = pd.read_sql_query(
        """
        SELECT *
        FROM users
        LIMIT 10
        """,
        conn
    )

display(customers)

,user_id,name,status,updated_at
0,U-00000,User_U-00000,Pending,2023-01-01 10:00:00
1,U-00000,User_U-00000,Active,2023-01-19 14:00:00
2,U-00000,User_U-00000,Suspended,2023-02-28 09:00:00
3,U-00001,User_U-00001,Pending,2023-01-01 10:00:00
4,U-00001,User_U-00001,Active,2023-01-28 14:00:00
5,U-00002,User_U-00002,Pending,2023-01-01 10:00:00
6,U-00002,User_U-00002,Active,2023-01-29 14:00:00
7,U-00002,User_U-00002,Suspended,2023-02-23 09:00:00
8,U-00003,User_U-00003,Pending,2023-01-01 10:00:00
9,U-00003,User_U-00003,Active,2023-01-22 14:00:00


In [6]:
with sqlite3.connect(DB_PATH) as conn:
    history = pd.read_sql_query(
        """
        SELECT *
        FROM users
        WHERE user_id = 'U-00000'
        ORDER BY updated_at DESC
        """,
        conn
    )

display(history)

,user_id,name,status,updated_at
0,U-00000,User_U-00000,Suspended,2023-02-28 09:00:00
1,U-00000,User_U-00000,Active,2023-01-19 14:00:00
2,U-00000,User_U-00000,Pending,2023-01-01 10:00:00


In [7]:
with sqlite3.connect(DB_PATH) as conn:
    ranked = pd.read_sql_query(
        """
        SELECT
            user_id,
            name,
            status,
            updated_at,
            ROW_NUMBER() OVER (
                PARTITION BY user_id
                ORDER BY updated_at DESC
            ) AS rn
        FROM users
        WHERE user_id = 'U-00000'
        """,
        conn
    )

display(ranked)


,user_id,name,status,updated_at,rn
0,U-00000,User_U-00000,Suspended,2023-02-28 09:00:00,1
1,U-00000,User_U-00000,Active,2023-01-19 14:00:00,2
2,U-00000,User_U-00000,Pending,2023-01-01 10:00:00,3


In [8]:
with sqlite3.connect(DB_PATH) as conn:
    latest_status = pd.read_sql_query(
        """
        WITH ranked_users AS (
            SELECT
                user_id,
                name,
                status,
                updated_at,
                ROW_NUMBER() OVER (
                    PARTITION BY user_id
                    ORDER BY updated_at DESC
                ) AS rn
            FROM users
        )
        SELECT user_id, name, status, updated_at
        FROM ranked_users
        WHERE rn = 1
          AND user_id = 'U-00000'
        """,
        conn
    )

display(latest_status)


,user_id,name,status,updated_at
0,U-00000,User_U-00000,Suspended,2023-02-28 09:00:00


In [9]:
with sqlite3.connect(DB_PATH) as conn:
    active_users = pd.read_sql_query(
        """
        WITH ranked_users AS (
            SELECT
                user_id,
                name,
                status,
                updated_at,
                ROW_NUMBER() OVER (
                    PARTITION BY user_id
                    ORDER BY updated_at DESC
                ) AS rn
            FROM users
        )
        SELECT
            user_id,
            name,
            status,
            updated_at
        FROM ranked_users
        WHERE rn = 1
          AND status = 'Active'
        """,
        conn
    )

display(active_users.head(10))

print("Latest Active customers:", len(active_users))
print("Unique customer IDs:", active_users["user_id"].nunique())

,user_id,name,status,updated_at
0,U-00001,User_U-00001,Active,2023-01-28 14:00:00
1,U-00003,User_U-00003,Active,2023-01-22 14:00:00
2,U-00005,User_U-00005,Active,2023-01-17 14:00:00
3,U-00006,User_U-00006,Active,2023-01-05 14:00:00
4,U-00007,User_U-00007,Active,2023-01-27 14:00:00
5,U-00008,User_U-00008,Active,2023-01-27 14:00:00
6,U-00011,User_U-00011,Active,2023-01-27 14:00:00
7,U-00015,User_U-00015,Active,2023-01-02 14:00:00
8,U-00017,User_U-00017,Active,2023-01-16 14:00:00
9,U-00019,User_U-00019,Active,2023-01-22 14:00:00


Latest Active customers: 14183
Unique customer IDs: 14183


In [10]:
from pathlib import Path
import pandas as pd

LOG_PATH = Path(r"C:\Users\maidk\Desktop\Reconciliation Task\EnterpriseReconciliation\server_logs.txt")

print("Log file exists:", LOG_PATH.exists())

logs = pd.read_csv(LOG_PATH)

print("Total log rows:", len(logs))

display(logs.head(5))

Log file exists: True
Total log rows: 100000


,Raw_Server_Log
0,[2023-03-15 00:00:00] INFO: Transaction proces...
1,[2023-09-09 00:00:00] ERROR: Transaction proce...
2,[2023-05-09 00:00:00] INFO: Transaction proces...
3,[2023-10-21 00:00:00] INFO: Transaction proces...
4,[2023-01-10 00:00:00] INFO: Transaction proces...


In [11]:
# Count records containing ERROR
error_mask = logs["Raw_Server_Log"].str.contains(
    "ERROR",
    na=False
)

error_logs = logs[error_mask]
successful_logs = logs[~error_mask]

print("Total logs:", len(logs))
print("Error logs:", len(error_logs))
print("Non-error logs:", len(successful_logs))

Total logs: 100000
Error logs: 5098
Non-error logs: 94902


In [12]:
print(successful_logs["Raw_Server_Log"].iloc[0])


[2023-03-15 00:00:00] INFO: Transaction processed. payload={"u":"U-19484", "item_code":"PRD-874", "eur_val":3404.44}


In [13]:
import re

log = successful_logs["Raw_Server_Log"].iloc[0]

user_id = re.search(r'"u":"(U-\d+)"', log)

print("User ID:", user_id.group(1))

User ID: U-19484


In [14]:
pattern = (
    r'^\[(?P<Date>\d{4}-\d{2}-\d{2})\s+\d{2}:\d{2}:\d{2}\].*?'
    r'payload=\{"u":"(?P<User_ID>U-\d+)",\s*'
    r'"item_code":"(?P<Product_ID>PRD-\d+)",\s*'
    r'"eur_val":(?P<EUR_Value>[\d.]+)\}'
)

result = pd.Series([log]).str.extract(pattern)

result["Date"] = pd.to_datetime(result["Date"])
result["EUR_Value"] = pd.to_numeric(result["EUR_Value"])

display(result)

,Date,User_ID,Product_ID,EUR_Value
0,2023-03-15,U-19484,PRD-874,3404.44


In [15]:
pattern = (
    r'^\[(?P<Date>\d{4}-\d{2}-\d{2})\s+\d{2}:\d{2}:\d{2}\].*?'
    r'payload=\{"u":"(?P<User_ID>U-\d+)",\s*'
    r'"item_code":"(?P<Product_ID>PRD-\d+)",\s*'
    r'"eur_val":(?P<EUR_Value>[\d.]+)\}'
)

transactions = successful_logs["Raw_Server_Log"].str.extract(pattern)

transactions["Date"] = pd.to_datetime(
    transactions["Date"], errors="coerce"
)

transactions["EUR_Value"] = pd.to_numeric(
    transactions["EUR_Value"], errors="coerce"
)

print("Total successful logs:", len(successful_logs))
print("Extracted transactions:", len(transactions))
print("Rows with missing extracted fields:", transactions.isna().any(axis=1).sum())

display(transactions.head(10))

Total successful logs: 94902
Extracted transactions: 94902
Rows with missing extracted fields: 0


,Date,User_ID,Product_ID,EUR_Value
0,2023-03-15,U-19484,PRD-874,3404.44
2,2023-05-09,U-18932,PRD-905,3406.21
3,2023-10-21,U-15882,PRD-284,164.08
4,2023-01-10,U-22781,PRD-665,26.80
5,2023-05-23,U-13348,PRD-589,4964.77
6,2023-10-20,U-22062,PRD-363,48.63
7,2023-12-02,U-22146,PRD-223,960.60
8,2023-03-21,U-08211,PRD-741,1129.34
9,2023-09-23,U-20854,PRD-315,2139.44
10,2023-08-04,U-11746,PRD-893,1529.29


In [16]:
from pathlib import Path
import pandas as pd

BASE_PATH = Path(
    r"C:\Users\maidk\Desktop\Reconciliation Task\EnterpriseReconciliation"
)

RATE_PATH = BASE_PATH / "daily_exchange_rates.csv"

rates = pd.read_csv(RATE_PATH)

print("Total exchange-rate rows:", len(rates))
print("\nColumn names:")
print(rates.columns.tolist())

print("\nFirst 10 rows:")
display(rates.head(10))

print("\nMissing values:")
print(rates.isna().sum())

Total exchange-rate rows: 365

Column names:
['Date', 'EUR_to_USD']

First 10 rows:


,Date,EUR_to_USD
0,2023-01-01,1.124836
1,2023-01-02,1.093087
2,2023-01-03,1.132384
3,2023-01-04,1.176151
4,2023-01-05,1.088292
5,2023-01-06,NaN
6,2023-01-07,NaN
7,2023-01-08,1.138372
8,2023-01-09,1.076526
9,2023-01-10,1.127128



Missing values:
Date            0
EUR_to_USD    104
dtype: int64


In [17]:
# Convert Date column to datetime
rates["Date"] = pd.to_datetime(rates["Date"])

# Sort dates in ascending order
rates = rates.sort_values("Date").reset_index(drop=True)

# Count missing rates before filling
missing_before = rates["EUR_to_USD"].isna().sum()

# Fill missing rates using the previous available rate
rates["EUR_to_USD"] = rates["EUR_to_USD"].ffill()

# Count missing rates after filling
missing_after = rates["EUR_to_USD"].isna().sum()

print("Missing rates before filling:", missing_before)
print("Missing rates after filling:", missing_after)

# Check the dates around the first missing values
display(rates.iloc[3:9])

Missing rates before filling: 104
Missing rates after filling: 0


,Date,EUR_to_USD
3,2023-01-04,1.176151
4,2023-01-05,1.088292
5,2023-01-06,1.088292
6,2023-01-07,1.088292
7,2023-01-08,1.138372
8,2023-01-09,1.076526


In [18]:
# Merge transactions with exchange rates using Date
transactions_with_rates = transactions.merge(
    rates,
    on="Date",
    how="left",
    validate="many_to_one"
)

print("Original transactions:", len(transactions))
print("Transactions after merge:", len(transactions_with_rates))

print(
    "Missing exchange rates after merge:",
    transactions_with_rates["EUR_to_USD"].isna().sum()
)

display(transactions_with_rates.head(10))


Original transactions: 94902
Transactions after merge: 94902
Missing exchange rates after merge: 0


,Date,User_ID,Product_ID,EUR_Value,EUR_to_USD
0,2023-03-15,U-19484,PRD-874,3404.44,1.178232
1,2023-05-09,U-18932,PRD-905,3406.21,1.104983
2,2023-10-21,U-15882,PRD-284,164.08,1.075350
3,2023-01-10,U-22781,PRD-665,26.80,1.127128
4,2023-05-23,U-13348,PRD-589,4964.77,1.019626
5,2023-10-20,U-22062,PRD-363,48.63,1.075350
6,2023-12-02,U-22146,PRD-223,960.60,1.066142
7,2023-03-21,U-08211,PRD-741,1129.34,1.000622
8,2023-09-23,U-20854,PRD-315,2139.44,1.048781
9,2023-08-04,U-11746,PRD-893,1529.29,1.084237


In [19]:
import sqlite3

conn = sqlite3.connect(DB_PATH)

active_users = pd.read_sql_query("""
    WITH ranked_users AS (
        SELECT
            user_id,
            status,
            ROW_NUMBER() OVER (
                PARTITION BY user_id
                ORDER BY updated_at DESC
            ) AS rn
        FROM users
    )
    SELECT user_id
    FROM ranked_users
    WHERE rn = 1
      AND status = 'Active'
""", conn)

conn.close()

active_users = active_users.rename(
    columns={"user_id": "User_ID"}
)

print("Latest Active customers:", len(active_users))
print("Unique Active customer IDs:", active_users["User_ID"].nunique())

display(active_users.head(10))

Latest Active customers: 14183
Unique Active customer IDs: 14183


,User_ID
0,U-00001
1,U-00003
2,U-00005
3,U-00006
4,U-00007
5,U-00008
6,U-00011
7,U-00015
8,U-00017
9,U-00019


In [20]:
eligible_transactions = transactions_with_rates.merge(
    active_users,
    on="User_ID",
    how="inner",
    validate="many_to_one"
)

print("Transactions before customer filtering:", len(transactions_with_rates))
print("Transactions for latest Active customers:", len(eligible_transactions))
print("Unique customers in final dataset:", eligible_transactions["User_ID"].nunique())
print("Unique products in final dataset:", eligible_transactions["Product_ID"].nunique())

display(eligible_transactions.head(10))

Transactions before customer filtering: 94902
Transactions for latest Active customers: 53899
Unique customers in final dataset: 13888
Unique products in final dataset: 900


,Date,User_ID,Product_ID,EUR_Value,EUR_to_USD
0,2023-03-15,U-19484,PRD-874,3404.44,1.178232
1,2023-10-21,U-15882,PRD-284,164.08,1.075350
2,2023-01-10,U-22781,PRD-665,26.80,1.127128
3,2023-10-20,U-22062,PRD-363,48.63,1.075350
4,2023-12-02,U-22146,PRD-223,960.60,1.066142
5,2023-03-21,U-08211,PRD-741,1129.34,1.000622
6,2023-02-01,U-12689,PRD-223,3390.14,1.192614
7,2023-07-18,U-06991,PRD-732,690.08,1.102910
8,2023-02-06,U-21283,PRD-842,4097.27,1.110443
9,2023-08-17,U-21759,PRD-177,2877.31,1.064235


In [21]:
# Calculate revenue in USD
eligible_transactions["USD_Revenue"] = (
    eligible_transactions["EUR_Value"]
    * eligible_transactions["EUR_to_USD"]
)

# Calculate total revenue in both currencies
total_eur = eligible_transactions["EUR_Value"].sum()
total_usd = eligible_transactions["USD_Revenue"].sum()

print(f"Total eligible EUR revenue: €{total_eur:,.2f}")
print(f"Total eligible USD revenue: ${total_usd:,.2f}")

print("\nFirst 10 transactions with USD revenue:")
display(
    eligible_transactions[
        ["Date", "User_ID", "Product_ID", "EUR_Value",
         "EUR_to_USD", "USD_Revenue"]
    ].head(10)
)

Total eligible EUR revenue: €134,958,287.20
Total eligible USD revenue: $148,274,622.91

First 10 transactions with USD revenue:


,Date,User_ID,Product_ID,EUR_Value,EUR_to_USD,USD_Revenue
0,2023-03-15,U-19484,PRD-874,3404.44,1.178232,4011.220772
1,2023-10-21,U-15882,PRD-284,164.08,1.075350,176.443420
2,2023-01-10,U-22781,PRD-665,26.80,1.127128,30.207030
3,2023-10-20,U-22062,PRD-363,48.63,1.075350,52.294268
4,2023-12-02,U-22146,PRD-223,960.60,1.066142,1024.135923
5,2023-03-21,U-08211,PRD-741,1129.34,1.000622,1130.041946
6,2023-02-01,U-12689,PRD-223,3390.14,1.192614,4043.128118
7,2023-07-18,U-06991,PRD-732,690.08,1.102910,761.096434
8,2023-02-06,U-21283,PRD-842,4097.27,1.110443,4549.785527
9,2023-08-17,U-21759,PRD-177,2877.31,1.064235,3062.133474


In [22]:
# Extract month from each transaction date
eligible_transactions["Month"] = (
    eligible_transactions["Date"].dt.to_period("M")
)

# Group transactions by month and calculate totals
monthly_revenue = (
    eligible_transactions
    .groupby("Month")
    .agg(
        Total_Transactions=("USD_Revenue", "size"),
        Total_EUR_Revenue=("EUR_Value", "sum"),
        Total_USD_Revenue=("USD_Revenue", "sum")
    )
    .reset_index()
)

# Format revenue to 2 decimal places for display
monthly_revenue["Total_EUR_Revenue"] = (
    monthly_revenue["Total_EUR_Revenue"].round(2)
)

monthly_revenue["Total_USD_Revenue"] = (
    monthly_revenue["Total_USD_Revenue"].round(2)
)

print("Monthly revenue report:")
display(monthly_revenue)

print("\nNumber of months:", len(monthly_revenue))
print(
    "Monthly USD totals:",
    round(monthly_revenue["Total_USD_Revenue"].sum(), 2)
)

Monthly revenue report:


,Month,Total_Transactions,Total_EUR_Revenue,Total_USD_Revenue
0,2023-01,4635,11807690.10,12841761.38
1,2023-02,4139,10515276.57,11436130.90
2,2023-03,4565,11292316.21,12348931.84
3,2023-04,4388,10975579.91,12015376.19
4,2023-05,4570,11378070.84,12333644.45
5,2023-06,4338,10849710.84,12084355.82
6,2023-07,4514,11278439.93,12444163.58
7,2023-08,4627,11688733.95,12812730.66
8,2023-09,4401,11043757.04,12228952.05
9,2023-10,4528,11233792.52,12501330.36



Number of months: 12
Monthly USD totals: 148274622.92


In [23]:
# Validate monthly transaction counts
print(
    "Eligible transaction count:",
    len(eligible_transactions)
)

print(
    "Monthly transaction count:",
    monthly_revenue["Total_Transactions"].sum()
)

# Compare revenue using full precision
monthly_usd_unrounded = (
    eligible_transactions
    .assign(Month=eligible_transactions["Date"].dt.to_period("M"))
    .groupby("Month")["USD_Revenue"]
    .sum()
    .sum()
)

print(
    "Full-precision eligible USD revenue:",
    round(eligible_transactions["USD_Revenue"].sum(), 2)
)

print(
    "Full-precision monthly USD revenue:",
    round(monthly_usd_unrounded, 2)
)

# Check whether monthly transaction counts match
assert (
    monthly_revenue["Total_Transactions"].sum()
    == len(eligible_transactions)
)

print("\nValidation passed: transaction counts match.")

Eligible transaction count: 53899
Monthly transaction count: 53899
Full-precision eligible USD revenue: 148274622.91
Full-precision monthly USD revenue: 148274622.91

Validation passed: transaction counts match.


In [24]:
# Calculate total revenue for each customer
customer_revenue = (
    eligible_transactions
    .groupby("User_ID")
    .agg(
        Total_Transactions=("USD_Revenue", "size"),
        Total_EUR_Revenue=("EUR_Value", "sum"),
        Total_USD_Revenue=("USD_Revenue", "sum")
    )
    .reset_index()
)

# Sort customers from highest to lowest USD revenue
customer_revenue = customer_revenue.sort_values(
    "Total_USD_Revenue",
    ascending=False
).reset_index(drop=True)

# Display the top 5 customers
top_5_customers = customer_revenue.head(5).copy()

top_5_customers["Total_EUR_Revenue"] = (
    top_5_customers["Total_EUR_Revenue"].round(2)
)

top_5_customers["Total_USD_Revenue"] = (
    top_5_customers["Total_USD_Revenue"].round(2)
)

print("Top 5 customers by USD revenue:")
display(top_5_customers)

print("\nTotal customers in revenue report:", len(customer_revenue))
print(
    "Unique customers in eligible transactions:",
    eligible_transactions["User_ID"].nunique()
)

Top 5 customers by USD revenue:


,User_ID,Total_Transactions,Total_EUR_Revenue,Total_USD_Revenue
0,U-06582,12,36542.40,40050.99
1,U-23313,11,35424.47,39555.81
2,U-03805,11,35880.78,39529.95
3,U-20018,8,33814.60,38649.97
4,U-06489,9,35639.75,38629.14



Total customers in revenue report: 13888
Unique customers in eligible transactions: 13888


In [25]:
# 1. Check required columns for missing values
required_columns = [
    "Date",
    "User_ID",
    "Product_ID",
    "EUR_Value",
    "EUR_to_USD",
    "USD_Revenue"
]

print("Missing values in required columns:")
print(eligible_transactions[required_columns].isna().sum())

# 2. Verify the USD revenue formula
expected_usd = (
    eligible_transactions["EUR_Value"]
    * eligible_transactions["EUR_to_USD"]
)

formula_matches = (
    eligible_transactions["USD_Revenue"]
    .round(8)
    .equals(expected_usd.round(8))
)

print("\nUSD formula matches:", formula_matches)

# 3. Compare overall revenue totals
total_usd = eligible_transactions["USD_Revenue"].sum()
monthly_usd = monthly_revenue["Total_USD_Revenue"].sum()
customer_usd = customer_revenue["Total_USD_Revenue"].sum()

print("\nOverall USD revenue:", round(total_usd, 2))
print("Monthly USD revenue (rounded rows):", round(monthly_usd, 2))
print("Customer USD revenue (full precision):", round(customer_usd, 2))

print("\nCustomer count matches:",
      len(customer_revenue) == eligible_transactions["User_ID"].nunique())

Missing values in required columns:
Date           0
User_ID        0
Product_ID     0
EUR_Value      0
EUR_to_USD     0
USD_Revenue    0
dtype: int64

USD formula matches: True

Overall USD revenue: 148274622.91
Monthly USD revenue (rounded rows): 148274622.92
Customer USD revenue (full precision): 148274622.91

Customer count matches: True


In [26]:
from pathlib import Path

OUTPUT_PATH = BASE_PATH / "output"
OUTPUT_PATH.mkdir(exist_ok=True)

# 1. Final eligible transaction details
eligible_transactions.to_csv(
    OUTPUT_PATH / "eligible_transactions.csv",
    index=False
)

# 2. Monthly report — preserve full precision
monthly_export = (
    eligible_transactions
    .assign(Month=eligible_transactions["Date"].dt.to_period("M"))
    .groupby("Month")
    .agg(
        Total_Transactions=("USD_Revenue", "size"),
        Total_EUR_Revenue=("EUR_Value", "sum"),
        Total_USD_Revenue=("USD_Revenue", "sum")
    )
    .reset_index()
)

monthly_export.to_csv(
    OUTPUT_PATH / "monthly_revenue_report.csv",
    index=False
)

# 3. All customers ranked by USD revenue
customer_revenue.to_csv(
    OUTPUT_PATH / "customer_revenue_report.csv",
    index=False
)

# 4. Top 5 customers
top_5_customers.to_csv(
    OUTPUT_PATH / "top_5_customers.csv",
    index=False
)

print("Export complete. Files created:")
for file_path in sorted(OUTPUT_PATH.glob("*.csv")):
    print(f"{file_path.name} — {file_path.stat().st_size:,} bytes")

Export complete. Files created:
customer_revenue_report.csv — 549,574 bytes
eligible_transactions.csv — 4,355,171 bytes
monthly_revenue_report.csv — 595 bytes
top_5_customers.csv — 210 bytes


In [27]:
# Verify exported CSV files
output_files = {
    "eligible_transactions.csv": 53899,
    "monthly_revenue_report.csv": 12,
    "customer_revenue_report.csv": 13888,
    "top_5_customers.csv": 5
}

for filename, expected_rows in output_files.items():
    file_path = OUTPUT_PATH / filename
    df_check = pd.read_csv(file_path)

    status = "PASS" if len(df_check) == expected_rows else "CHECK"

    print(
        f"{filename}: {len(df_check):,} rows "
        f"(expected {expected_rows:,}) — {status}"
    )

print("\nVerification complete.")

eligible_transactions.csv: 53,899 rows (expected 53,899) — PASS
monthly_revenue_report.csv: 12 rows (expected 12) — PASS
customer_revenue_report.csv: 13,888 rows (expected 13,888) — PASS
top_5_customers.csv: 5 rows (expected 5) — PASS

Verification complete.


In [28]:

from pathlib import Path
import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.chart import BarChart, LineChart, Reference
from openpyxl.utils import get_column_letter

# 1. Project folders
BASE_PATH = Path(
    r"C:\Users\maidk\Desktop\Reconciliation Task\EnterpriseReconciliation"
)

OUTPUT_PATH = BASE_PATH / "output"
OUTPUT_PATH.mkdir(exist_ok=True)

# 2. Read CSV files
monthly = pd.read_csv(OUTPUT_PATH / "monthly_revenue_report.csv")
top5 = pd.read_csv(OUTPUT_PATH / "top_5_customers.csv")
customers = pd.read_csv(OUTPUT_PATH / "customer_revenue_report.csv")
transactions = pd.read_csv(OUTPUT_PATH / "eligible_transactions.csv")

print("All CSV files loaded successfully.")

# 3. Create Excel workbook
wb = Workbook()
dashboard = wb.active
dashboard.title = "Dashboard"

navy = "17365D"
blue = "1F4E78"
light_blue = "D9EAF7"

# Dashboard heading
dashboard.merge_cells("A1:H1")
dashboard["A1"] = "ENTERPRISE DATA RECONCILIATION DASHBOARD"
dashboard["A1"].font = Font(
    bold=True, size=16, color="FFFFFF"
)
dashboard["A1"].fill = PatternFill("solid", fgColor=navy)
dashboard["A1"].alignment = Alignment(horizontal="center")
dashboard.row_dimensions[1].height = 32

# 4. Dashboard KPIs
kpis = [
    ("A3:B3", "A4:B5", "ELIGIBLE TRANSACTIONS",
     len(transactions), "0"),
    ("C3:D3", "C4:D5", "UNIQUE CUSTOMERS",
     transactions["User_ID"].nunique(), "0"),
    ("E3:F3", "E4:F5", "UNIQUE PRODUCTS",
     transactions["Product_ID"].nunique(), "0"),
    ("G3:H3", "G4:H5", "TOTAL USD REVENUE",
     transactions["USD_Revenue"].sum(), '$#,##0.00')
]

for label_range, value_range, label, value, fmt in kpis:
    dashboard.merge_cells(label_range)
    dashboard.merge_cells(value_range)

    label_cell = dashboard[label_range.split(":")[0]]
    value_cell = dashboard[value_range.split(":")[0]]

    label_cell.value = label
    label_cell.font = Font(bold=True, color="FFFFFF")
    label_cell.fill = PatternFill("solid", fgColor=blue)
    label_cell.alignment = Alignment(horizontal="center")

    value_cell.value = float(value) if fmt.startswith("$") else int(value)
    value_cell.font = Font(bold=True, size=14, color=navy)
    value_cell.fill = PatternFill("solid", fgColor=light_blue)
    value_cell.alignment = Alignment(horizontal="center")
    value_cell.number_format = fmt

# 5. Function to add CSV data as Excel sheets
def add_data_sheet(sheet_name, df):
    ws = wb.create_sheet(sheet_name)

    # Header row
    for col_num, column_name in enumerate(df.columns, start=1):
        cell = ws.cell(1, col_num, str(column_name))
        cell.font = Font(bold=True, color="FFFFFF")
        cell.fill = PatternFill("solid", fgColor=blue)
        cell.alignment = Alignment(wrap_text=True)

    # Data rows
    for row_num, row in enumerate(
        df.itertuples(index=False, name=None), start=2
    ):
        for col_num, value in enumerate(row, start=1):
            if pd.isna(value):
                value = None
            elif hasattr(value, "item"):
                try:
                    value = value.item()
                except (ValueError, AttributeError):
                    pass

            ws.cell(row_num, col_num, value)

    ws.freeze_panes = "A2"
    ws.auto_filter.ref = ws.dimensions
    ws.sheet_view.showGridLines = False

    # Column widths and number formatting
    for col_num, col_name in enumerate(df.columns, start=1):
        ws.column_dimensions[
            get_column_letter(col_num)
        ].width = min(max(len(str(col_name)) + 4, 14), 28)

        if "Revenue" in str(col_name) or col_name == "EUR_Value":
            for r in range(2, ws.max_row + 1):
                ws.cell(r, col_num).number_format = '#,##0.00'

        if col_name == "EUR_to_USD":
            for r in range(2, ws.max_row + 1):
                ws.cell(r, col_num).number_format = '0.000000'

    return ws

# 6. Add detailed sheets
ws_monthly = add_data_sheet("Monthly Revenue", monthly)
ws_top5 = add_data_sheet("Top 5 Customers", top5)
ws_customers = add_data_sheet("Customer Revenue", customers)
ws_transactions = add_data_sheet("Eligible Transactions", transactions)

# 7. Monthly USD revenue column chart
month_col = monthly.columns.get_loc("Month") + 1
usd_col = monthly.columns.get_loc("Total_USD_Revenue") + 1

chart1 = BarChart()
chart1.type = "col"
chart1.style = 10
chart1.title = "Monthly USD Revenue"
chart1.y_axis.title = "Revenue (USD)"
chart1.x_axis.title = "Month"

chart1.add_data(
    Reference(ws_monthly, min_col=usd_col,
              min_row=1, max_row=len(monthly) + 1),
    titles_from_data=True
)
chart1.set_categories(
    Reference(ws_monthly, min_col=month_col,
              min_row=2, max_row=len(monthly) + 1)
)
chart1.height = 8
chart1.width = 16
chart1.legend = None

dashboard.add_chart(chart1, "A7")

# 8. Monthly transaction volume line chart
txn_col = monthly.columns.get_loc("Total_Transactions") + 1

chart2 = LineChart()
chart2.style = 13
chart2.title = "Monthly Transaction Volume"
chart2.y_axis.title = "Transactions"
chart2.x_axis.title = "Month"

chart2.add_data(
    Reference(ws_monthly, min_col=txn_col,
              min_row=1, max_row=len(monthly) + 1),
    titles_from_data=True
)
chart2.set_categories(
    Reference(ws_monthly, min_col=month_col,
              min_row=2, max_row=len(monthly) + 1)
)
chart2.height = 8
chart2.width = 16
chart2.legend = None

dashboard.add_chart(chart2, "I7")

# 9. Top 5 customers revenue chart
customer_id_col = top5.columns.get_loc("User_ID") + 1
customer_usd_col = top5.columns.get_loc("Total_USD_Revenue") + 1

chart3 = BarChart()
chart3.type = "bar"
chart3.style = 11
chart3.title = "Top 5 Customers by USD Revenue"
chart3.x_axis.title = "Revenue (USD)"
chart3.y_axis.title = "Customer"

chart3.add_data(
    Reference(ws_top5, min_col=customer_usd_col,
              min_row=1, max_row=len(top5) + 1),
    titles_from_data=True
)
chart3.set_categories(
    Reference(ws_top5, min_col=customer_id_col,
              min_row=2, max_row=len(top5) + 1)
)
chart3.height = 8
chart3.width = 16
chart3.legend = None

dashboard.add_chart(chart3, "A23")

# 10. Dashboard formatting
for col, width in {
    "A": 20, "B": 16, "C": 20, "D": 16,
    "E": 20, "F": 16, "G": 20, "H": 16, "I": 4
}.items():
    dashboard.column_dimensions[col].width = width

dashboard.freeze_panes = "A3"
dashboard.sheet_view.showGridLines = False

# 11. Save Excel workbook
excel_path = OUTPUT_PATH / "Enterprise_Data_Reconciliation_Dashboard.xlsx"
wb.save(excel_path)

print("\nDashboard generated successfully!")
print("File location:", excel_path)
print("Eligible transactions:", len(transactions))
print("Unique customers:", transactions["User_ID"].nunique())
print("Unique products:", transactions["Product_ID"].nunique())
print("Total USD revenue: ${:,.2f}".format(
    transactions["USD_Revenue"].sum()
))


All CSV files loaded successfully.

Dashboard generated successfully!
File location: C:\Users\maidk\Desktop\Reconciliation Task\EnterpriseReconciliation\output\Enterprise_Data_Reconciliation_Dashboard.xlsx
Eligible transactions: 53899
Unique customers: 13888
Unique products: 900
Total USD revenue: $148,274,622.91
